# 7-3절 연습 문제 풀이

이 노트북은 7-3절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch07/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 7-3/7-4절 공통 - MNIST 오토인코더
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
DATA_ROOT = '../../downloads'

def mnist(train=True):
    return datasets.MNIST(root=DATA_ROOT, train=train, download=True,
                          transform=transforms.ToTensor())

class MNISTAutoEncoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(),
                                     nn.Linear(128, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 128), nn.ReLU(),
                                     nn.Linear(128, 784), nn.Sigmoid())
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z).view(-1, 1, 28, 28), z

def train_ae(model, epochs=5, lr=1e-3, noise=0.0):
    loader = DataLoader(mnist(), batch_size=128, shuffle=True)
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for e in range(1, epochs + 1):
        model.train(); tot = n = 0
        for x, _ in loader:
            x = x.to(device)
            inp = (x + torch.randn_like(x) * noise).clamp(0, 1) if noise else x
            out, _ = model(inp)
            loss = criterion(out, x)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tot += loss.item() * len(x); n += len(x)
        print(f'  {e}/{epochs} 복원 손실 {tot / n:.5f}')
    return model

## 연습 7-10

잠재 벡터의 크기를 4, 8, 16, 32로 늘려가면서 오토인코더 모델을 학습하고 재현 결과 및 노이즈 제거 결과를 확인해 보자. 잠재 벡터의 크기에 따라 어떤 변화를 관찰할 수 있는가?

In [ ]:
test_x = torch.stack([mnist(False)[i][0] for i in range(8)]).to(device)
for latent in (4, 8, 16, 32):
    torch.manual_seed(SEED)
    print(f'잠재 벡터 크기 {latent}')
    model = train_ae(MNISTAutoEncoder(latent), epochs=5)
    model.eval()
    with torch.no_grad():
        recon, _ = model(test_x)
        noisy = (test_x + torch.randn_like(test_x) * 0.3).clamp(0, 1)
        denoised, _ = model(noisy)
    viz.plot_images(list(test_x.cpu()) + list(recon.cpu()) + list(denoised.cpu()),
                    ['원본'] * 8 + [f'복원({latent})'] * 8 + ['노이즈 제거'] * 8,
                    images_per_row=8)

잠재 벡터가 클수록 복원 품질은 좋아진다. 담을 수 있는 정보가 늘기 때문이다. 다만 지나치게 크면 **압축이라는 목적이 흐려지고**, 입력을 거의 그대로 통과시켜 노이즈까지 복원해 버려 노이즈 제거 성능은 오히려 떨어진다. 압축률과 복원 품질 사이의 트레이드오프다.

## 연습 7-11

MNISTAutoEncoder 모델 때문에 오토인코더 속의 인코더와 디코더의 구조가 대칭을 이뤄야 한다고 오해할 수도 있다. 다음과 같이 구조를 수정한 3개의 오토인코더 모델을 만들어 보고 각각 결과를 확인해 보자. 필요하다면 잠재 벡터의 크기를 수정해도 좋다.

인코더에 포함된 선형 계층을 2개로 수정

디코더에 포함된 선형 계층을 2개로 수정

인코더를 합성곱 신경망의 구조로 변경: 이 경우 마지막 합성곱 계층이 출력하는 특징 지도를 평탄화해서 1차원의 잠재 벡터로 변환해야 한다.

In [ ]:
class DeepEncoderAE(nn.Module):      # 인코더만 선형 계층 2개
    def __init__(self, latent_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
                                     nn.Linear(256, 64), nn.ReLU(),
                                     nn.Linear(64, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 128), nn.ReLU(),
                                     nn.Linear(128, 784), nn.Sigmoid())
    def forward(self, x):
        z = self.encoder(x); return self.decoder(z).view(-1, 1, 28, 28), z

class DeepDecoderAE(nn.Module):      # 디코더만 선형 계층 추가
    def __init__(self, latent_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(),
                                     nn.Linear(128, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(),
                                     nn.Linear(64, 256), nn.ReLU(),
                                     nn.Linear(256, 784), nn.Sigmoid())
    def forward(self, x):
        z = self.encoder(x); return self.decoder(z).view(-1, 1, 28, 28), z

class ConvEncoderAE(nn.Module):      # 인코더를 합성곱으로
    def __init__(self, latent_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, 2, 1), nn.ReLU(),      # 28 -> 14
            nn.Conv2d(16, 32, 3, 2, 1), nn.ReLU(),     # 14 -> 7
            nn.Flatten(), nn.Linear(32 * 7 * 7, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 128), nn.ReLU(),
                                     nn.Linear(128, 784), nn.Sigmoid())
    def forward(self, x):
        z = self.encoder(x); return self.decoder(z).view(-1, 1, 28, 28), z

for name, cls in [('인코더 2계층', DeepEncoderAE), ('디코더 확장', DeepDecoderAE),
                  ('합성곱 인코더', ConvEncoderAE)]:
    torch.manual_seed(SEED)
    print(f'[{name}]')
    train_ae(cls(), epochs=5)

인코더와 디코더는 **대칭일 필요가 없다**. 각자 맡은 일이 다르기 때문이다. 압축이 어려운 데이터라면 인코더를 깊게, 복원이 어렵다면 디코더를 깊게 만들면 된다. 합성곱 인코더는 이미지의 공간 구조를 활용해 같은 잠재 크기에서도 더 좋은 복원 품질을 낸다.

## 연습 7-12

[도전 문제] [연습 문제 7-11]에서 세 번째 항목으로 제안한 합성곱 계층으로 구성된 인코더처럼 디코더도 합성곱 계층을 사용해 만들 수 있다. 그런데 nn.Conv2d로 만든 합성곱 계층은 특징 지도의 크기를 늘릴 수 없어 합성곱 디코더를 만들 수 없다.

파이토치를 사용해 합성곱 디코더를 만들 때 특징 지도의 크기를 늘리는 데 사용되는 대표적인 두 가지 방법이 있다. 하나는 계층을 지날수록 정보를 확장하며 이미지를 복원하는 전치 합성곱 계층(nn.ConvTranspose2d 클래스로 생성)을 사용하는 방법이며, 다른 하나는 보간법으로 이미지의 크기를 늘리는 업샘플링 계층(nn.Upsample 클래스로 생성)을 nn.Conv2d로 만든 합성곱 계층과 함께 사용하는 방법이다.

In [ ]:
# 특징 지도를 키우는 방법: ConvTranspose2d(역합성곱) 또는 Upsample + Conv2d
class ConvAutoEncoder(nn.Module):
    def __init__(self, latent_ch=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, 2, 1), nn.ReLU(),          # 28 -> 14
            nn.Conv2d(16, latent_ch, 3, 2, 1), nn.ReLU())  # 14 -> 7
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(latent_ch, 16, 3, 2, 1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, 2, 1, output_padding=1), nn.Sigmoid())
    def forward(self, x):
        z = self.encoder(x); return self.decoder(z), z

x = torch.randn(1, 1, 28, 28)
m = ConvAutoEncoder()
print(f'인코더 출력(잠재): {tuple(m.encoder(x).shape)}')
print(f'디코더 출력       : {tuple(m(x)[0].shape)}')
torch.manual_seed(SEED)
train_ae(ConvAutoEncoder(), epochs=5)

특징 지도를 키우는 방법은 두 가지다.

1. **`nn.ConvTranspose2d`(역합성곱)**: 합성곱의 역연산으로 크기를 키운다. `output_padding`으로 정확한 크기를 맞춘다. 11장 VAE·DCGAN의 디코더가 이 방식이다.
2. **`nn.Upsample` + `nn.Conv2d`**: 먼저 보간으로 키운 뒤 합성곱을 적용한다. 역합성곱의 격자 무늬(체커보드) 현상을 피할 수 있다.